# Release watch: the notebook version

The starting point of the tutorial. It works, once, on my laptop, when I remember to run it.

The question: *did any package I depend on ship something this week that I need to act on?*

In [1]:
import os
from datetime import datetime, timedelta, timezone

import httpx

PACKAGES = ["prefect", "pydantic", "httpx", "fastmcp"]
since = datetime.now(timezone.utc) - timedelta(days=7)

## 1. What's new on PyPI?

In [2]:
new = []
for name in PACKAGES:
    data = httpx.get(f"https://pypi.org/pypi/{name}/json").json()
    for version, files in data["releases"].items():
        if files and datetime.fromisoformat(files[0]["upload_time_iso_8601"]) > since:
            new.append((name, version))

new

[('prefect', '3.8.7'),
 ('prefect', '3.8.7.dev5'),
 ('prefect', '3.8.7.dev6'),
 ('prefect', '3.8.7.dev7'),
 ('fastmcp', '4.0.10'),
 ('fastmcp', '4.0.6'),
 ('fastmcp', '4.0.7'),
 ('fastmcp', '4.0.8'),
 ('fastmcp', '4.0.9')]

Already some problems: prereleases and yanked versions sneak in, one slow PyPI response stalls everything, and there are no retries.

## 2. Ask a model what changed

In [3]:
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("Set ANTHROPIC_API_KEY to run this cell.")
else:
    client = Anthropic()
    name, version = new[0]
    reply = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        messages=[{"role": "user", "content": f"What changed in {name} {version}? Anything breaking?"}],
    )
    print(reply.content[0].text)

Set ANTHROPIC_API_KEY to run this cell.


## What this notebook can't do

- **Run on its own.** It only runs when I open it.
- **Survive a bad afternoon.** One timeout and the whole cell fails.
- **Avoid paying twice.** Rerun it and every release is sent to the model again.
- **Return anything a program can use.** The answer is free text, and the model is guessing from its training data rather than reading the release notes.
- **Show me what happened last Tuesday.** There's no history.

The rest of the tutorial fixes each of these by moving the code into `src/release_watch/` and wrapping it in Prefect.